# 01 — Main YouTube Data Collection (Target: 2,500 Comments)

**Continuation of the original `01_data_collection.ipynb`.** This version keeps the same YouTube Data API collection approach and the two-person Excel annotation template, while scaling collection beyond the earlier 50-comment pilot.

**Workflow:** configuration → video-source manifest → API retrieval → cleaning / deduplication → review → annotation export.

- Run each cell in order from your **project's `notebooks/` folder** using the `fyp-sarcasm` kernel.
- Place your YouTube API key in the project-root `.env` file as `YOUTUBE_API_KEY=...`. Never put the key in this notebook or GitHub.
- Add **at least 30–40 suitable video URLs** across diverse topics to `data/video_manifest_main.csv` before collecting the full dataset. The five starter URLs are only examples from your existing notebook.
- Only top-level YouTube comments are collected. We do **not** download author names, profiles, avatars or channel IDs.
- Raw and annotation data are research data: keep them private. Consult university ethics guidance and YouTube API data-use requirements.
- Labels are assigned **only by humans**; the script does not predict sarcasm.

`TARGET_USABLE=2500` is an **annotation candidate target**, not a guarantee of 2,500 high-quality labeled instances; you should manually review and document exclusions before annotation.

In [ ]:
from pathlib import Path
from urllib.parse import urlparse, parse_qs
import os, re, json, unicodedata
import pandas as pd
from dotenv import load_dotenv

# Keep the original notebook's project-path convention.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env", override=True)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"
ANNOTATION_DIR = PROJECT_ROOT / "data" / "annotated"
for directory in [RAW_DIR, CLEAN_DIR, ANNOTATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = PROJECT_ROOT / "data" / "video_manifest_main.csv"
MASTER_RAW_PATH = RAW_DIR / "main_youtube_comments.csv"
CLEAN_PATH = CLEAN_DIR / "main_comments_candidates.csv"
REPORT_PATH = CLEAN_DIR / "main_collection_report.json"

TARGET_USABLE = 2500
MAX_COMMENTS_PER_VIDEO = 120  # Enforces video diversity; ~30+ videos recommended.
BATCH_NAME = "main"          # Do not reuse 'batch01' from pilots.
CREATE_ANNOTATION_WORKBOOK = False  # Change to True only after auditing the collected data.

print("Project root:", PROJECT_ROOT)
print("Target annotation candidates:", TARGET_USABLE)

In [ ]:
# A source manifest makes sampling choices explicit and reproducible.
# Update topic, include, and ordering after reviewing each video.
STARTER_URLS = [
    "https://www.youtube.com/watch?v=MYh5VCBuPt4",
    "https://www.youtube.com/watch?v=d6t-ck9wOr4",
    "https://www.youtube.com/watch?v=t3xTTnAuzdY",
    "https://youtu.be/5uoCo0Wc5iI?list=RD5uoCo0Wc5iI",
    "https://www.youtube.com/watch?v=g_ndubZzE3Q",
]

if not MANIFEST_PATH.exists():
    starter = pd.DataFrame({
        "video_url": STARTER_URLS,
        "topic": ["review_required"] * len(STARTER_URLS),
        "include": ["yes"] * len(STARTER_URLS),
        "order": ["time", "relevance", "time", "relevance", "time"],
    })
    starter.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")
    print("Starter manifest created:", MANIFEST_PATH)

manifest = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("")
required = {"video_url", "topic", "include", "order"}
missing = required - set(manifest.columns)
if missing:
    raise ValueError(f"Manifest missing columns: {missing}")

manifest = manifest[manifest["include"].str.strip().str.lower().isin(["yes", "y", "true", "1"])].copy()
manifest["order"] = manifest["order"].str.strip().str.lower().replace("", "time")
if (~manifest["order"].isin(["time", "relevance"])).any():
    raise ValueError("Manifest order must be either 'time' or 'relevance'")
manifest = manifest.drop_duplicates(subset=["video_url"], keep="first").reset_index(drop=True)

print(f"Included video URLs: {len(manifest)}")
if len(manifest) * MAX_COMMENTS_PER_VIDEO < TARGET_USABLE:
    print("WARNING: Too few video URLs to reach the target under the per-video cap.")
if (manifest["topic"] == "review_required").any():
    print("WARNING: Please replace 'review_required' with actual topic categories.")
display(manifest.head(10))

In [ ]:
# Pure helper functions — no API calls in this cell.
from urllib.parse import urlparse, parse_qs

RAW_COLUMNS = ["comment_id", "text", "video_id", "video_link", "topic", "sampling_order"]
TIMESTAMP_ONLY = re.compile(r"^(?:\d{1,2}:)?\d{1,2}:\d{2}(?:\s*[-–,;.]\s*)?$")
URL_ONLY = re.compile(r"^https?://\S+$", re.IGNORECASE)

def extract_video_id(url):
    url = str(url).strip()
    parsed = urlparse(url)
    host = parsed.netloc.lower().removeprefix("www.")
    if host == "youtu.be":
        video_id = parsed.path.strip("/").split("/")[0]
    elif host in {"youtube.com", "m.youtube.com"}:
        if parsed.path == "/watch":
            video_id = parse_qs(parsed.query).get("v", [""])[0]
        else:
            parts = parsed.path.strip("/").split("/")
            video_id = parts[1] if len(parts) >= 2 and parts[0] in {"shorts", "live"} else ""
    else:
        video_id = ""
    if not re.fullmatch(r"[A-Za-z0-9_-]{11}", video_id):
        raise ValueError(f"Invalid YouTube video URL: {url}")
    return video_id

def normalize_text(s):
    return re.sub(r"\s+", " ", str(s) if pd.notna(s) else "").strip()

def text_key(s):
    return unicodedata.normalize("NFKC", normalize_text(s)).casefold()

def is_unusable(s):
    t = normalize_text(s)
    return not t or bool(TIMESTAMP_ONLY.fullmatch(t)) or bool(URL_ONLY.fullmatch(t))

def load_previous_pilot_data():
    # Avoid contamination from pilot files still in the original project.
    # Excludes the growing 'main' dataset itself.
    paths = list(RAW_DIR.glob("batch*.csv")) + list(RAW_DIR.glob("pilot*.csv"))
    paths += list(ANNOTATION_DIR.glob("batch*_annotation_template.csv"))
    previous_ids, previous_texts = set(), set()
    for path in set(paths):
        try:
            old = pd.read_csv(path, dtype=str, keep_default_na=False)
            if "comment_id" in old:
                previous_ids.update(old["comment_id"].dropna().astype(str))
            if "text" in old:
                previous_texts.update(text_key(x) for x in old["text"] if normalize_text(x))
        except (OSError, UnicodeError, pd.errors.ParserError) as exc:
            print(f"WARNING: Could not read existing pilot file {path.name}: {exc}")
    return previous_ids, previous_texts

def make_candidates(raw, previous_ids, previous_texts):
    if raw.empty:
        return pd.DataFrame(columns=RAW_COLUMNS + ["needs_manual_review"])
    df = raw.copy()
    for col in RAW_COLUMNS:
        if col not in df:
            df[col] = ""
    df = df[RAW_COLUMNS].fillna("")
    df["text"] = df["text"].map(normalize_text)
    df = df[~df["text"].map(is_unusable)].copy()
    df = df[~df["comment_id"].isin(previous_ids)].copy()
    df["text_key"] = df["text"].map(text_key)
    df = df[~df["text_key"].isin(previous_texts)].copy()
    df = df.drop_duplicates(subset=["comment_id"], keep="first")
    # Only text is a model input: prevent identical text across videos from leaking across splits.
    df = df.drop_duplicates(subset=["text_key"], keep="first")
    # Flag rather than automatically discard brief or emoji-only comments.
    df["needs_manual_review"] = df["text"].map(lambda x: len(x) < 5 or not any(ch.isalpha() for ch in x))
    return df.drop(columns=["text_key"]).reset_index(drop=True)

# Fail early for malformed URLs without needing to spend API quota.
manifest["video_id"] = manifest["video_url"].map(extract_video_id)
manifest = manifest.drop_duplicates(subset=["video_id"]).reset_index(drop=True)
previous_ids, previous_texts = load_previous_pilot_data()
print(f"Existing pilot exclusions: {len(previous_ids)} comment IDs, {len(previous_texts)} unique texts")

## Collect comments (uses your YouTube API key)

This cell continues from `data/raw/main_youtube_comments.csv` on later runs. The raw master is updated **after each video**, so a failure does not discard previously saved videos. It stops when the cleaned candidate count reaches 2,500 or when there are no more usable sources.

For a safer initial check, temporarily set `TARGET_USABLE = 200`, run the notebook, review the results, then restore 2,500 and add more source URLs to the manifest.

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

api_key = os.getenv("YOUTUBE_API_KEY")
if not api_key:
    raise ValueError("Missing YOUTUBE_API_KEY in the project-root .env file")
youtube = build("youtube", "v3", developerKey=api_key, cache_discovery=False)

if MASTER_RAW_PATH.exists():
    raw = pd.read_csv(MASTER_RAW_PATH, dtype=str, keep_default_na=False)
    for col in RAW_COLUMNS:
        if col not in raw:
            raw[col] = ""
    raw = raw[RAW_COLUMNS].drop_duplicates(subset=["comment_id"], keep="first")
else:
    raw = pd.DataFrame(columns=RAW_COLUMNS)

def save_raw(df):
    tmp = MASTER_RAW_PATH.with_suffix(".tmp")
    df.to_csv(tmp, index=False, encoding="utf-8-sig")
    tmp.replace(MASTER_RAW_PATH)

def fetch_video_comments(video_id, topic, order="time", limit=120):
    records, next_page = [], None
    while len(records) < limit:
        request = youtube.commentThreads().list(
            part="snippet", videoId=video_id,
            maxResults=min(100, limit - len(records)),
            pageToken=next_page, textFormat="plainText", order=order
        )
        response = request.execute()
        for item in response.get("items", []):
            comment = item.get("snippet", {}).get("topLevelComment", {})
            snippet = comment.get("snippet", {})
            if comment.get("id"):
                records.append({
                    "comment_id": comment["id"],
                    "text": snippet.get("textDisplay", ""),
                    "video_id": video_id,
                    "video_link": f"https://www.youtube.com/watch?v={video_id}",
                    "topic": topic,
                    "sampling_order": order,
                })
        next_page = response.get("nextPageToken")
        if not next_page or not response.get("items"):
            break
    return records

for row in manifest.itertuples(index=False):
    candidates = make_candidates(raw, previous_ids, previous_texts)
    if len(candidates) >= TARGET_USABLE:
        print(f"Target reached: {len(candidates)} usable candidates")
        break
    current_count = (raw["video_id"].astype(str) == row.video_id).sum() if len(raw) else 0
    if current_count >= MAX_COMMENTS_PER_VIDEO:
        print(f"Skipping {row.video_id}: already collected {current_count}")
        continue
    try:
        batch = fetch_video_comments(row.video_id, row.topic, row.order, MAX_COMMENTS_PER_VIDEO)
    except HttpError as exc:
        message = str(exc)
        print(f"Could not collect {row.video_id}: {message[:220]}")
        if "quotaexceeded" in message.lower() or "dailylimitexceeded" in message.lower():
            print("API quota exhausted. Saved progress can be resumed on the next run.")
            break
        continue
    if batch:
        raw = pd.concat([raw, pd.DataFrame(batch, columns=RAW_COLUMNS)], ignore_index=True)
        raw = raw.drop_duplicates(subset=["comment_id"], keep="first")[RAW_COLUMNS]
        save_raw(raw)
    candidates = make_candidates(raw, previous_ids, previous_texts)
    print(f"{row.video_id} ({row.topic}): fetched {len(batch)} | total candidate count {len(candidates)}")

# Always save the clean snapshot and summary at the end of this run.
candidates = make_candidates(raw, previous_ids, previous_texts)
candidates = candidates.head(TARGET_USABLE).reset_index(drop=True)
candidates.to_csv(CLEAN_PATH, index=False, encoding="utf-8-sig")
summary = {
    "target_candidates": TARGET_USABLE,
    "raw_unique_comment_ids": int(len(raw)),
    "candidate_count": int(len(candidates)),
    "candidate_short_or_emoji_only_needs_review": int(candidates["needs_manual_review"].sum()),
    "number_of_source_videos": int(candidates["video_id"].nunique()),
    "topics": {str(k): int(v) for k, v in candidates["topic"].value_counts().items()},
    "pilot_ids_excluded_from_candidate_pool": len(previous_ids),
    "sampling_note": "Top-level comments; source video and order metadata retained; not a random sample of all YouTube comments.",
}
REPORT_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved:", MASTER_RAW_PATH)
print("Saved:", CLEAN_PATH)
print("Saved:", REPORT_PATH)
print(json.dumps(summary, indent=2, ensure_ascii=False))
display(candidates.head(10))

## Quality-control checkpoint

Check counts and a random sample before asking annotators to start. For ambiguous cases, keep the original text and the source-video link; manually decide whether the record is usable. Do **not** pre-select comments because they look sarcastic, as that would change the sampling distribution.

**Important:** Collecting 2,500 candidates does not establish that 2,500 valid binary labels will remain after `Uncertain` decisions and adjudication. Additional collection may be needed.

In [ ]:
if candidates.empty:
    print("No candidate comments collected yet. Review the manifest and API access.")
else:
    print("Candidate count:", len(candidates))
    print("Videos:", candidates["video_id"].nunique())
    print("Manual review flags:", int(candidates["needs_manual_review"].sum()))
    print("Topic distribution:")
    display(candidates["topic"].value_counts().rename_axis("topic").reset_index(name="count"))
    print("Comments per video (top 10):")
    display(candidates["video_id"].value_counts().head(10))
    print("Random review sample (inspect text and video context):")
    display(candidates.sample(n=min(20, len(candidates)), random_state=42)[["comment_id", "text", "video_link", "needs_manual_review"]])

## Generate the annotation workbook (preserves original two-sheet design)

This cell **does not run unless** `CREATE_ANNOTATION_WORKBOOK = True` in the first code cell. Do this only after reviewing candidates and deciding that the source mix and cleaning are adequate.

- Both annotators get identical comment IDs, text, and video links.
- `Sincere`, `Sarcasm/Irony`, `Uncertain` are annotation options. `Uncertain` is a review/exclusion label, **not** a model class; confirm this follows your final locked guidelines.
- Keep both independent annotation sheets unchanged until agreement is calculated and disagreements adjudicated in a **separate** file.
- If the output already exists, the notebook will **refuse to overwrite it**, protecting your labels.

In [ ]:
if not CREATE_ANNOTATION_WORKBOOK:
    print("Annotation export is OFF. When the dataset is ready, set CREATE_ANNOTATION_WORKBOOK=True and run this cell.")
else:
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.worksheet.datavalidation import DataValidation

    if candidates.empty:
        raise ValueError("No candidates found. Collect and clean comments first.")

    annotation_source = candidates.reset_index(drop=True)
    annotation_df = pd.DataFrame({
        "ID": [f"{BATCH_NAME}-{i:05d}" for i in range(1, len(annotation_source) + 1)],
        "text": annotation_source["text"],
        "video_link": annotation_source["video_link"],
        "label": "",
        "confidence": "",
        "notes": "",
    })

    # Preserve IDs for later joins with raw data; never publish publicly without review.
    id_lookup = annotation_source[["comment_id", "video_id", "topic"]].copy()
    id_lookup.insert(0, "ID", annotation_df["ID"])
    lookup_path = ANNOTATION_DIR / "main_annotation_id_lookup_PRIVATE.csv"
    csv_path = ANNOTATION_DIR / "main_annotation_template.csv"
    output_path = ANNOTATION_DIR / "main_annotation_v1.xlsx"

    if output_path.exists() or csv_path.exists() or lookup_path.exists():
        raise FileExistsError("An annotation export already exists. Move/archive it deliberately before exporting again; never overwrite labeled work.")

    annotation_df.to_csv(csv_path, index=False, encoding="utf-8-sig")
    id_lookup.to_csv(lookup_path, index=False, encoding="utf-8-sig")
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for person in ["Person_1", "Person_2"]:
            annotation_df.to_excel(writer, sheet_name=person, index=False)

    wb = load_workbook(output_path)
    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for cell in ws[1]:
            cell.fill = PatternFill("solid", fgColor="17365D")
            cell.font = Font(color="FFFFFF", bold=True)
        for col, width in {"A": 18, "B": 75, "C": 48, "D": 22, "E": 22, "F": 48}.items():
            ws.column_dimensions[col].width = width

        label_dv = DataValidation(type="list", formula1='"Sincere,Sarcasm/Irony,Uncertain"', allow_blank=True)
        confidence_dv = DataValidation(type="list", formula1='"1 - Low,2 - Medium,3 - High"', allow_blank=True)
        ws.add_data_validation(label_dv)
        ws.add_data_validation(confidence_dv)
        label_dv.add(f"D2:D{ws.max_row}")
        confidence_dv.add(f"E2:E{ws.max_row}")
        for cells in ws.iter_rows(min_row=2):
            for cell in cells:
                cell.alignment = Alignment(vertical="top", wrap_text=True)
            # Do not let text beginning with =, +, -, @ execute as Excel formulas.
            cells[1].data_type = "s"
            if cells[2].value:
                cells[2].hyperlink = cells[2].value
                cells[2].style = "Hyperlink"
            ws.row_dimensions[cells[0].row].height = 38
    wb.save(output_path)
    print("Created protected annotation workbook:", output_path)
    print("Also created private ID lookup:", lookup_path)
    print("Rows for each annotator:", len(annotation_df))

### Important methodological notes

1. **Sampling:** the source manifest determines which videos and topics enter the dataset. `order=time` and `order=relevance` are platform-provided ranking orders, **not random sampling**. Record this limitation in the dissertation.
2. **Pilot independence:** candidate filtering excludes comments and exact normalized text found in previously stored `batch*` pilot CSVs; pilot materials saved elsewhere must be added to the exclusion check manually.
3. **Cleaning:** blanks, timestamp-only strings, link-only strings, repeated comment IDs, and identical normalized text are excluded. Emojis, slang, punctuation and mixed Vietnamese/English comments are preserved.
4. **Review:** shorter and emoji-only comments are flagged, not automatically labeled as bad. Review spam, advertisements, inappropriate content, and language suitability consistently before generating annotation sheets.
5. **Human labels:** two independent annotators first; three-category Cohen's kappa (if `Uncertain` is allowed) before adjudication; separate final binary dataset (`0=Sincere`, `1=Sarcasm/Irony`) after ambiguous cases are handled.
6. **Ethics and platform requirements:** keep `.env` out of Git, avoid publishing raw comments/identifiers, minimize personal data, and follow applicable consent, research ethics, and YouTube API policies.